## Upload Corpus to Hugging Face Datasets

Uploads the shared corpus (abstracts by category, PDF-derived chunks by category, DPRA pipeline artifacts) to a private Hugging Face Datasets repo so both collaborators can pull the same data from Colab or local.

**Design note**: this script is written to be run independently by either collaborator. It only uploads whatever category files exist locally, it does not assume all 10 categories are present. Running it twice (once per person, on their own categories) is safe: Hugging Face repos are commit-based, so each run adds/updates files without deleting anyone else's prior uploads. If you re-run this after regenerating a file, it creates a new commit rather than silently overwriting history, so previous versions stay recoverable.

**What gets uploaded**:
1. Abstract files present under `data/arxiv/abstracts/by_category/*.jsonl`
2. PDF-chunk files under `data/chunks/<category>/*.jsonl`, consolidated into one file per category before upload (source files stay untouched on disk)
3. DPRA pipeline artifacts: `chunk_hashes.json`, `category_counts.csv`, `page_type_counts.csv`, `routing_ledger.jsonl`

**Before running**: you need a Hugging Face account and a private dataset repo already created (or this notebook will create one for you on first run, see the config cell).

### Install dependencies (Colab only, skip if running locally with them already installed)

In [2]:
# Uncomment if running in Colab or a fresh environment
!pip install huggingface_hub -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuration

Update `REPO_ID` to your actual dataset repo (format `username/dataset-name` or `org-name/dataset-name`). Both collaborators should point at the same `REPO_ID`.

In [3]:
import os

# --- Repo settings ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"  
REPO_TYPE = "dataset"
PRIVATE = False 

# --- Local paths, relative to this notebook's location under notebooks/ ---
PROJECT_ROOT = ".."  # adjust if this notebook moves
ABSTRACTS_BY_CATEGORY_DIR = os.path.join(PROJECT_ROOT, "data", "arxiv", "abstracts", "by_category")
CHUNKS_DIR = os.path.join(PROJECT_ROOT, "data", "chunks")

# The 10 target categories, used only to validate what we find, not to require all of them
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# DPRA pipeline artifact files, expected directly under CHUNKS_DIR
PIPELINE_ARTIFACTS = [
    "chunk_hashes.json",
    "category_counts.csv",
    "page_type_counts.csv",
    "routing_ledger.jsonl",
]

# Where consolidated PDF-chunk files get written locally before upload (scratch space, not tracked in git)
CONSOLIDATED_CHUNKS_DIR = os.path.join(PROJECT_ROOT, "data", "chunks", "_consolidated_for_upload")
os.makedirs(CONSOLIDATED_CHUNKS_DIR, exist_ok=True)

### Authenticate

Prompts for a Hugging Face access token (needs write access) if you're not already logged in. In Colab, this shows a login widget. Locally, run `huggingface-cli login` once beforehand instead, and this cell will just confirm you're authenticated.

In [4]:
from huggingface_hub import login, whoami

try:
    user_info = whoami()
    print(f"Already authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Already authenticated as: Sakhiur


### Create the repo if it doesn't exist yet

Safe to run every time. If the repo already exists, this is a no-op (`exist_ok=True`). Only the first person to run this actually creates it; the second person's run just confirms it's already there.

In [5]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

create_repo(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    private=PRIVATE,
    exist_ok=True,
)

print(f"Repo ready: https://huggingface.co/datasets/{REPO_ID}")

Repo ready: https://huggingface.co/datasets/Sakhiur/empirical-rag-paradigm-benchmark


### Discover what's available locally

Reports which categories are present for abstracts and for PDF chunks, so you can see what this run will actually upload before it happens. Missing categories are expected, not an error, since your collaborator's categories may not be on your machine.

In [6]:
# Abstracts: one file per category expected, e.g. cs.AI.jsonl
found_abstract_categories = []
for category in ALL_CATEGORIES:
    path = os.path.join(ABSTRACTS_BY_CATEGORY_DIR, f"{category}.jsonl")
    if os.path.exists(path):
        found_abstract_categories.append(category)

# PDF chunks: a folder per category, containing many per-paper jsonl files
found_chunk_categories = []
for category in ALL_CATEGORIES:
    folder = os.path.join(CHUNKS_DIR, category)
    if os.path.isdir(folder):
        n_files = len([f for f in os.listdir(folder) if f.endswith(".jsonl")])
        if n_files > 0:
            found_chunk_categories.append((category, n_files))

print("Abstract categories found locally:")
for c in found_abstract_categories:
    print(f"  {c}")
missing_abstracts = [c for c in ALL_CATEGORIES if c not in found_abstract_categories]
if missing_abstracts:
    print(f"  (not found, presumably your collaborator's: {missing_abstracts})")

print("\nPDF chunk categories found locally:")
for c, n in found_chunk_categories:
    print(f"  {c}: {n} per-paper files")
found_chunk_cat_names = [c for c, _ in found_chunk_categories]
missing_chunks = [c for c in ALL_CATEGORIES if c not in found_chunk_cat_names]
if missing_chunks:
    print(f"  (not found, presumably your collaborator's: {missing_chunks})")

print("\nPipeline artifacts found:")
for artifact in PIPELINE_ARTIFACTS:
    path = os.path.join(CHUNKS_DIR, artifact)
    status = "found" if os.path.exists(path) else "MISSING"
    print(f"  {artifact}: {status}")

Abstract categories found locally:
  cs.AI
  cs.LG
  cs.IR
  cs.DB
  cs.SE
  (not found, presumably your collaborator's: ['cs.CV', 'cs.CL', 'cs.NE', 'cs.DC', 'cs.CR'])

PDF chunk categories found locally:
  cs.AI: 498 per-paper files
  cs.LG: 498 per-paper files
  cs.IR: 499 per-paper files
  cs.DB: 498 per-paper files
  cs.SE: 500 per-paper files
  (not found, presumably your collaborator's: ['cs.CV', 'cs.CL', 'cs.NE', 'cs.DC', 'cs.CR'])

Pipeline artifacts found:
  chunk_hashes.json: found
  category_counts.csv: found
  page_type_counts.csv: found
  routing_ledger.jsonl: found


### Consolidate PDF chunks, one file per category

My PDF chunks are currently one JSONL per paper (e.g. `data/chunks/cs.AI/1709.02256.jsonl`). For a clean dataset on the Hub, each category becomes a single JSONL file with all its papers' page-chunks concatenated, matching the abstract files' structure. This only reads and re-writes to a scratch folder, it does not touch or delete my original per-paper files.

Each line keeps its original fields as produced by DPRA (`paper_id`, `category`, `page_num`, `page_type`, `text`, `char_count`, `content_hash`, `is_duplicate`, `source_path`), nothing is dropped or renamed.

In [7]:
import json

consolidated_paths = {}

for category, n_files in found_chunk_categories:
    source_folder = os.path.join(CHUNKS_DIR, category)
    out_path = os.path.join(CONSOLIDATED_CHUNKS_DIR, f"{category}.jsonl")

    per_paper_files = sorted(
        f for f in os.listdir(source_folder) if f.endswith(".jsonl")
    )

    lines_written = 0
    malformed = 0
    with open(out_path, "w", encoding="utf-8") as out_f:
        for fname in per_paper_files:
            fpath = os.path.join(source_folder, fname)
            with open(fpath, "r", encoding="utf-8") as in_f:
                for line in in_f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        json.loads(line)  # validate only, write the original line as-is
                    except json.JSONDecodeError:
                        malformed += 1
                        print(f"[WARN] {fname}: malformed line, skipping")
                        continue
                    out_f.write(line + "\n")
                    lines_written += 1

    consolidated_paths[category] = out_path
    print(f"{category}: {len(per_paper_files)} papers -> {lines_written} page-chunks written ({malformed} malformed skipped) -> {out_path}")

cs.AI: 498 papers -> 9394 page-chunks written (0 malformed skipped) -> ..\data\chunks\_consolidated_for_upload\cs.AI.jsonl
cs.LG: 498 papers -> 9783 page-chunks written (0 malformed skipped) -> ..\data\chunks\_consolidated_for_upload\cs.LG.jsonl
cs.IR: 499 papers -> 6769 page-chunks written (0 malformed skipped) -> ..\data\chunks\_consolidated_for_upload\cs.IR.jsonl
cs.DB: 498 papers -> 9044 page-chunks written (0 malformed skipped) -> ..\data\chunks\_consolidated_for_upload\cs.DB.jsonl
cs.SE: 500 papers -> 8221 page-chunks written (0 malformed skipped) -> ..\data\chunks\_consolidated_for_upload\cs.SE.jsonl


### Upload abstract files

Uploaded under `abstracts/by_category/<category>.jsonl` in the repo. `commit_message` records who uploaded what and when, visible in the repo's commit history on the Hub.

In [8]:
for category in found_abstract_categories:
    local_path = os.path.join(ABSTRACTS_BY_CATEGORY_DIR, f"{category}.jsonl")
    repo_path = f"abstracts/by_category/{category}.jsonl"

    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        commit_message=f"Upload abstracts for {category} (by {user_info['name']})",
    )
    print(f"Uploaded {repo_path}")

Processing Files (1 / 1): 100%|██████████| 20.5MB / 20.5MB, 1.71MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded abstracts/by_category/cs.AI.jsonl


Processing Files (1 / 1): 100%|██████████| 32.0MB / 32.0MB, 2.84MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded abstracts/by_category/cs.LG.jsonl


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded abstracts/by_category/cs.IR.jsonl


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded abstracts/by_category/cs.DB.jsonl


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded abstracts/by_category/cs.SE.jsonl


### Upload consolidated PDF chunk files

Uploaded under `pdf_chunks/<category>.jsonl`.

In [ ]:
for category, local_path in consolidated_paths.items():
    repo_path = f"pdf_chunks/{category}.jsonl"

    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        commit_message=f"Upload PDF chunks for {category} (by {user_info['name']})",
    )
    print(f"Uploaded {repo_path}")

Processing Files (1 / 1): 100%|██████████| 34.3MB / 34.3MB, 1.43MB/s  
New Data Upload: 100%|██████████| 34.3MB / 34.3MB, 1.43MB/s  


Uploaded pdf_chunks/cs.AI.jsonl


Processing Files (1 / 1): 100%|██████████| 36.6MB / 36.6MB, 1.79MB/s  
New Data Upload: 100%|██████████| 36.6MB / 36.6MB, 1.79MB/s  


Uploaded pdf_chunks/cs.LG.jsonl


Processing Files (1 / 1): 100%|██████████| 30.5MB / 30.5MB, 1.66MB/s  
New Data Upload: 100%|██████████| 30.5MB / 30.5MB, 1.66MB/s  


Uploaded pdf_chunks/cs.IR.jsonl


Processing Files (1 / 1): 100%|██████████| 42.1MB / 42.1MB, 2.35MB/s  
New Data Upload: 100%|██████████| 42.1MB / 42.1MB, 2.35MB/s  


Uploaded pdf_chunks/cs.DB.jsonl


Processing Files (0 / 0): |          |  0.00B /  0.00B            

### Upload DPRA pipeline artifacts

Uploaded under `pipeline_artifacts/`. These are shared, category-spanning files (not per-category), so whoever runs this second will simply overwrite them with a newer commit if they've changed. Worth agreeing verbally on who's the source of truth for these if both of you regenerate them independently, to avoid one person's run clobbering the other's in a way that loses information neither of you can recover from a diff.

In [ ]:
for artifact in PIPELINE_ARTIFACTS:
    local_path = os.path.join(CHUNKS_DIR, artifact)
    if not os.path.exists(local_path):
        print(f"[SKIP] {artifact}: not found locally")
        continue

    repo_path = f"pipeline_artifacts/{artifact}"
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        commit_message=f"Upload {artifact} (by {user_info['name']})",
    )
    print(f"Uploaded {repo_path}")

### Verify: list everything currently in the repo

Confirms what's actually on the Hub right now, combining both collaborators' uploads if both have run this. Good sanity check before either of you starts building on top of it (embeddings, clustering, QA generation), to make sure you're both seeing the same thing.

In [ ]:
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)

abstracts_on_hub = sorted(f for f in repo_files if f.startswith("abstracts/"))
chunks_on_hub = sorted(f for f in repo_files if f.startswith("pdf_chunks/"))
artifacts_on_hub = sorted(f for f in repo_files if f.startswith("pipeline_artifacts/"))

print(f"Repo: https://huggingface.co/datasets/{REPO_ID}\n")

print(f"Abstract files on Hub ({len(abstracts_on_hub)}):")
for f in abstracts_on_hub:
    print(f"  {f}")

print(f"\nPDF chunk files on Hub ({len(chunks_on_hub)}):")
for f in chunks_on_hub:
    print(f"  {f}")

print(f"\nPipeline artifacts on Hub ({len(artifacts_on_hub)}):")
for f in artifacts_on_hub:
    print(f"  {f}")

abstract_categories_on_hub = {f.split("/")[-1].replace(".jsonl", "") for f in abstracts_on_hub}
chunk_categories_on_hub = {f.split("/")[-1].replace(".jsonl", "") for f in chunks_on_hub}

missing_abstract_cats = set(ALL_CATEGORIES) - abstract_categories_on_hub
missing_chunk_cats = set(ALL_CATEGORIES) - chunk_categories_on_hub

print(f"\nStill missing from abstracts (across both collaborators): {sorted(missing_abstract_cats) if missing_abstract_cats else 'none, all 10 present'}")
print(f"Still missing from PDF chunks (across both collaborators): {sorted(missing_chunk_cats) if missing_chunk_cats else 'none, all 10 present'}")

### How to load this back later (reference, not run here)

For the next notebook (embeddings, clustering, QA generation), pull files like this from either Colab or local. `hf_hub_download` caches locally, so repeated calls across sessions do not re-download unless the file changed on the Hub.

In [ ]:
from huggingface_hub import hf_hub_download

example_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename="abstracts/by_category/cs.AI.jsonl",  # change to whatever category you need
)
print(f"Downloaded to: {example_path}")

with open(example_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
print(f"First record preview: {first_line[:200]}...")